# Stufe 2 — Erkennung: das Feld `text` füllen

**KI-gestützte Dokumentenaufbereitung · Referenz-Implementierung · Niveau DQR 5/6**

Stufe 1 hat eine Seite in typisierte, verortete und geordnete Bausteine zerlegt und
das Ergebnis als `SeitenBefund` abgelegt. Kein einziger Baustein trägt Text.

Dieses Notebook füllt ihn. Es liest `befunde/*_stufe1.json`, schneidet für jeden
Block einen Ausschnitt aus der hoch aufgelösten Seite, schickt ihn mit dem passenden
Prompt an PaddleOCR-VL-1.5 und legt das Ergebnis als `befunde/*_stufe2.json` ab.

## Einordnung ins Phasenmodell

| Phase | Inhalt | in diesem Notebook |
|---|---|---|
| 01b | Layout-Analyse | erledigt — wird eingelesen |
| **01c** | **Erkennung je Region** | **Schwerpunkt** |
| 02 | Datenmodellierung | Erweiterung des Schemas |
| 03 | Prompting | ein festes Vokabular, keine freie Anweisung |
| 04–08 | Konsolidierung bis Monitoring | spätere Stufen |

## Was dieses Notebook bewusst nicht tut

- **Es entscheidet nichts über Bedeutung.** `<fcel>…<nl>` wird unverändert
  durchgereicht, nicht in eine Tabelle umgebaut. Das ist Stufe 4.
- **Es korrigiert nichts.** Wenn das Modell etwas falsch liest, steht es falsch in
  der JSON. Korrektur setzt Vergleich voraus, Vergleich setzt eine zweite Quelle
  voraus — auch Stufe 4.
- **Es beschreibt keine Bilder.** `image` und Verwandte bekommen keinen Aufruf; ihre
  Ausschnitte werden als PNG abgelegt und in Stufe 4 vom Vision-LLM typisiert.

## Voraussetzungen

LM Studio läuft mit geladenem PaddleOCR-VL-1.5 und aktiver Bildunterstützung. Falls
das noch nicht geprüft ist: erst das Sondierungs-Notebook durchlaufen lassen. Ohne
dessen vier Antworten misst dieses Notebook nur, wie schnell etwas Falsches entsteht.

```bash
pip install requests pymupdf pydantic opencv-python-headless numpy
```

In [1]:
from __future__ import annotations

import base64
import json
import re
import time
from enum import Enum
from pathlib import Path
from typing import Literal

import cv2
import numpy as np
import pymupdf
import requests
from pydantic import BaseModel, Field, model_validator

# --- Anzupassen
LMS_URL   = "http://localhost:1234/v1"
MODELL_ID = None                       # None = erstes geladenes Modell

BEFUND_DIR = Path("befunde")
UPLOADS    = Path("uploads")
AUSSCHNITT_DIR = Path("ausschnitte")

SEITEN = {
    "zimbardo":    UPLOADS / "Zimbardo_Psychologie_18Aufl_Extracted.pdf",
    "wahrnehmung": UPLOADS / "Wahrnehmungspsychologie_Extracted.pdf",
    "tietze":      UPLOADS / "HalbleiterSchaltungstechnik_TietzeSchenk_2002_Extracted.pdf",
}

PROBE_DPI      = 300      # Ausschnitte hoeher aufloesen als die Layout-Analyse
RAND_ANTEIL    = 0.12     # Rand als Anteil der Boxhoehe - siehe Abschnitt 3
RAND_MIN_PX    = 3        # untere Schranke, bei 200 dpi gemessen
RAND_MAX_PX    = 16       # obere Schranke
MAX_LUECKE_PX  = 40       # waagerechter Abstand, bis zu dem zusammengefuehrt wird
MIN_UEBERLAPP  = 0.5      # senkrechte Ueberlappung, ab der zusammengefuehrt wird

AUSSCHNITT_DIR.mkdir(exist_ok=True)

for name, pfad in SEITEN.items():
    befund = BEFUND_DIR / f"{name}_stufe1.json"
    print(f"{name:12s} PDF {'ok ' if pfad.exists() else 'FEHLT'}   "
          f"Befund {'ok' if befund.exists() else 'FEHLT'}")

zimbardo     PDF ok    Befund ok
wahrnehmung  PDF ok    Befund ok
tietze       PDF ok    Befund ok


---
## 1. Das Schema erweitern

Vier neue Felder in `Block`. Jedes beantwortet eine Frage, die spätestens in Stufe 4
gestellt wird und dann nicht mehr beantwortbar wäre.

| Feld | Frage |
|---|---|
| `text_format` | Wie ist `text` zu lesen — Klartext, LaTeX, OTSL, Datenreihe? |
| `text_quelle` | Wer hat den Text erzeugt? |
| `zusammengefuehrt_aus` | Aus welchen Blöcken der Stufe 1 ist dieser entstanden? |
| `ausschnitt` | Wo liegt das PNG, falls kein Text erzeugt wurde? |

`text_format` ist das wichtigste davon. Ohne dieses Feld müsste Stufe 4 den Inhalt
**raten** — `<fcel>` erkennt man noch, aber ob `\(x\)` aus einer abgesetzten oder
einer inline stehenden Formel stammt, steht nirgends im Text. Genau die Sorte
Information, die man in der Stufe erfassen muss, in der sie noch bekannt ist.

`text_quelle` neben dem bestehenden `quelle` zu führen, ist kein Versehen. `quelle`
sagt, woher der **Block** kommt (der Detektor), `text_quelle`, woher der **Text**
kommt (das VLM). Ab Stufe 4 können beide auseinanderfallen, etwa wenn ein Mensch
einen Text korrigiert, den Block aber stehen lässt.

> 💡 In einem ausgewachsenen Projekt gehörte dieses Schema in ein Modul, das alle
> Notebooks importieren. Hier steht es inline, damit das Notebook allein lauffähig
> bleibt — mit dem bekannten Preis, dass eine Änderung an zwei Stellen gepflegt
> werden muss.

In [2]:
class Ursprung(str, Enum):
    OBEN_LINKS = "oben_links"
    UNTEN_LINKS = "unten_links"


class Bezugsrahmen(str, Enum):
    MODELL_800 = "modell_800"
    BILD_PIXEL = "bild_pixel"
    SEITE_PUNKT = "seite_punkt"
    NORMIERT_1000 = "normiert_1000"


class Strom(str, Enum):
    HAUPT = "haupt"
    MARGINALIE = "marginalie"
    BOILERPLATE = "boilerplate"
    APPARAT = "apparat"


BOILERPLATE = {"header", "footer", "header_image", "footer_image", "number"}
APPARAT = {"footnote", "vision_footnote", "reference", "reference_content"}

def strom_fuer(pp_label: str) -> Strom:
    if pp_label in BOILERPLATE:  return Strom.BOILERPLATE
    if pp_label == "aside_text": return Strom.MARGINALIE
    if pp_label in APPARAT:      return Strom.APPARAT
    return Strom.HAUPT


class Bbox(BaseModel):
    x0: float
    y0: float
    x1: float
    y1: float
    rahmen: Bezugsrahmen
    ursprung: Ursprung = Ursprung.OBEN_LINKS

    @model_validator(mode="after")
    def _sortiert(self) -> "Bbox":
        if self.x1 < self.x0 or self.y1 < self.y0:
            raise ValueError(f"Bbox nicht sortiert: {self.x0},{self.y0} - {self.x1},{self.y1}")
        return self

    @property
    def breite(self) -> float:  return self.x1 - self.x0
    @property
    def hoehe(self) -> float:   return self.y1 - self.y0


class Block(BaseModel):
    id: int
    query_id: int
    pp_label: str
    score: float = Field(ge=0.0, le=1.0)
    bbox: Bbox
    lese_index: int | None = None
    polygon: list[tuple[float, float]] | None = None
    quelle: Literal["pp_doclayout", "pymupdf", "vlm", "mensch"] = "pp_doclayout"
    text: str | None = None

    # --- neu ab Stufe 2
    text_format: Literal["klartext", "latex", "otsl", "datenreihe"] | None = Field(
        default=None, description="Wie `text` zu lesen ist. None = kein Text erzeugt.")
    text_quelle: Literal["paddleocr_vl", "pymupdf", "mensch"] | None = Field(
        default=None, description="Wer den Text erzeugt hat - unabhängig von `quelle`.")
    zusammengefuehrt_aus: list[int] | None = Field(
        default=None, description="Block-ids der Stufe 1, falls zusammengeführt.")
    ausschnitt: str | None = Field(
        default=None, description="Pfad zum PNG, wenn kein Text erzeugt wurde.")

    @property
    def strom(self) -> Strom:
        return strom_fuer(self.pp_label)


class Lesekante(BaseModel):
    von: int
    nach: int
    konfidenz: float = Field(ge=0.0, le=1.0)
    marge: float | None = None


class SeitenBefund(BaseModel):
    quelle_datei: str
    seite: int = Field(ge=0)
    seite_breite_pt: float
    seite_hoehe_pt: float
    render_dpi: int
    bild_breite_px: int
    bild_hoehe_px: int
    bloecke: list[Block] = Field(default_factory=list)
    kanten: list[Lesekante] = Field(default_factory=list)
    warnungen: list[str] = Field(default_factory=list)

    def lesefolge(self, strom: Strom = Strom.HAUPT) -> list[Block]:
        erlaubt = {b.id: b for b in self.bloecke if b.strom is strom}
        if not erlaubt:
            return []
        if all(b.lese_index is not None for b in erlaubt.values()):
            return sorted(erlaubt.values(), key=lambda b: (b.lese_index, b.id))
        return sorted(erlaubt.values(), key=lambda b: b.id)


print("Schema geladen.")

Schema geladen.


---
## 2. Benachbarte Blöcke zusammenführen

Die Sondierung hat einen Fehler zutage gefördert, der ohne sie unbemerkt geblieben
wäre. Die Fußzeile der Zimbardo-Seite besteht aus **drei** Blöcken, und die Grenzen
verlaufen **mitten durch Wörter**. Das VLM las aus dem angeschnittenen Rest
`Inplar` — es sagt nicht „unlesbar", es rät.

Ein Detektor darf das. Er lokalisiert Regionen, nicht Wörter, und eine Region mit
scharfer Kante durch eine Buchstabenmitte ist geometrisch völlig in Ordnung. Der
Fehler entsteht erst dadurch, dass wir daraus einen Ausschnitt machen.

### Die Regel

Zwei Blöcke werden zusammengeführt, wenn **alle vier** Bedingungen gelten:

1. gleicher **Strom** — ein Marginalienblock verschmilzt nie mit dem Haupttext,
2. beide Labels sind **textartig** — Tabellen, Diagramme und Bilder bleiben unberührt,
3. die **senkrechte Überlappung** beträgt mindestens 50 % der kleineren Höhe,
4. die **waagerechte Lücke** ist kleiner als 40 px (bei 200 dpi).

Die Gruppierung läuft über eine Union-Find-Struktur, weil sich Zusammengehörigkeit
fortpflanzt: berührt A das B und B das C, gehören alle drei zusammen, auch wenn A
und C weit auseinanderliegen.

> ⚠️ Beim Zusammenführen geht das **Polygon** verloren. Die Vereinigung zweier
> Umrisse ist kein Umriss, und ein falsches Polygon ist schlechter als keins. Das
> zusammengeführte Rechteck bleibt, die Feinkontur nicht.

In [3]:
TEXTARTIG = {
    "text", "paragraph_title", "doc_title", "abstract", "aside_text", "footnote",
    "vision_footnote", "figure_title", "content", "header", "footer", "number",
    "reference", "reference_content", "algorithm", "vertical_text", "formula_number",
}


def _v_ueberlappung(a: Bbox, b: Bbox) -> float:
    """Senkrechte Überlappung, bezogen auf die kleinere der beiden Höhen."""
    oben, unten = max(a.y0, b.y0), min(a.y1, b.y1)
    if unten <= oben:
        return 0.0
    return (unten - oben) / max(1e-6, min(a.hoehe, b.hoehe))


def _h_luecke(a: Bbox, b: Bbox) -> float:
    """Waagerechter Abstand. 0, wenn sich die Boxen überlappen."""
    if a.x1 < b.x0:  return b.x0 - a.x1
    if b.x1 < a.x0:  return a.x0 - b.x1
    return 0.0


def gruppieren(bloecke: list[Block]) -> list[list[int]]:
    """Indizes zusammengehöriger Blöcke. Union-Find, weil Nachbarschaft transitiv ist."""
    n = len(bloecke)
    eltern = list(range(n))

    def finde(i: int) -> int:
        while eltern[i] != i:
            eltern[i] = eltern[eltern[i]]      # Pfadverkuerzung
            i = eltern[i]
        return i

    def vereine(i: int, j: int) -> None:
        a, b = finde(i), finde(j)
        if a != b:
            eltern[max(a, b)] = min(a, b)

    for i in range(n):
        for j in range(i + 1, n):
            a, b = bloecke[i], bloecke[j]
            if a.strom is not b.strom:
                continue
            if a.pp_label not in TEXTARTIG or b.pp_label not in TEXTARTIG:
                continue
            if _v_ueberlappung(a.bbox, b.bbox) < MIN_UEBERLAPP:
                continue
            if _h_luecke(a.bbox, b.bbox) > MAX_LUECKE_PX:
                continue
            vereine(i, j)

    gruppen: dict[int, list[int]] = {}
    for i in range(n):
        gruppen.setdefault(finde(i), []).append(i)
    return [sorted(g) for g in gruppen.values()]


def zusammenfuehren(befund: SeitenBefund) -> SeitenBefund:
    """Neuer Befund mit zusammengeführten Blöcken. Der alte bleibt unberührt."""
    gruppen = gruppieren(befund.bloecke)
    gruppen.sort(key=lambda g: min(befund.bloecke[i].lese_index if
                                   befund.bloecke[i].lese_index is not None
                                   else befund.bloecke[i].id for i in g))

    neu: list[Block] = []
    abbildung: dict[int, int] = {}          # alte Block-id -> neue Block-id

    for neue_id, gruppe in enumerate(gruppen):
        mitglieder = [befund.bloecke[i] for i in gruppe]
        for m in mitglieder:
            abbildung[m.id] = neue_id

        if len(mitglieder) == 1:
            block = mitglieder[0].model_copy(update={"id": neue_id})
            neu.append(block)
            continue

        # Der breiteste Block gibt Label und Query vor - er trägt am meisten Inhalt.
        leit = max(mitglieder, key=lambda b: b.bbox.breite)
        neu.append(Block(
            id=neue_id,
            query_id=leit.query_id,
            pp_label=leit.pp_label,
            score=min(m.score for m in mitglieder),      # konservativ
            bbox=Bbox(
                x0=min(m.bbox.x0 for m in mitglieder), y0=min(m.bbox.y0 for m in mitglieder),
                x1=max(m.bbox.x1 for m in mitglieder), y1=max(m.bbox.y1 for m in mitglieder),
                rahmen=Bezugsrahmen.BILD_PIXEL),
            lese_index=min((m.lese_index for m in mitglieder
                            if m.lese_index is not None), default=None),
            polygon=None,                                # Vereinigung waere gelogen
            zusammengefuehrt_aus=[m.id for m in mitglieder],
        ))

    # Kanten umschreiben: interne Kanten fallen weg, doppelte werden pessimistisch vereint
    kanten: dict[tuple[int, int], Lesekante] = {}
    for k in befund.kanten:
        v, n = abbildung.get(k.von), abbildung.get(k.nach)
        if v is None or n is None or v == n:
            continue
        vorhanden = kanten.get((v, n))
        if vorhanden is None or (k.marge is not None and vorhanden.marge is not None
                                 and k.marge < vorhanden.marge):
            kanten[(v, n)] = Lesekante(von=v, nach=n, konfidenz=k.konfidenz, marge=k.marge)

    warnungen = list(befund.warnungen)
    for b in neu:
        if b.zusammengefuehrt_aus:
            warnungen.append(
                f"Blöcke {b.zusammengefuehrt_aus} zu #{b.id} ({b.pp_label}) zusammengeführt.")

    return befund.model_copy(update={
        "bloecke": neu, "kanten": list(kanten.values()), "warnungen": warnungen})

---
## 3. Ausschnitte mit proportionalem Rand

Der zweite Fund der Sondierung: ein fester Rand von 8 px ist für einen Fließtextblock
unauffällig und für eine 31 px hohe Seitenzahl ein Viertel der Bildhöhe. Prompt gegen
Nachbartext ist ein ungleicher Kampf.

Also skaliert der Rand mit der Boxhöhe — 12 %, gedeckelt nach unten und oben. Ein
zweizeiliger Block bekommt wenig, ein Absatz mehr, und keiner bekommt so viel, dass
die Nachbarspalte hineinragt.

Die Faustregel aus Notebook 1 bleibt: **Layout auf der verkleinerten Kopie,
Ausschnitte aus dem Original.** Die Boxen liegen bei 200 dpi, geschnitten wird bei 300.

In [4]:
class Seitenbild:
    """Hält die hoch aufgelöste Seite und schneidet Blöcke daraus."""

    def __init__(self, pdf: Path, befund: SeitenBefund, dpi: int = PROBE_DPI):
        dok = pymupdf.open(pdf)
        self.pg = dok[befund.seite]
        pix = self.pg.get_pixmap(dpi=dpi)
        roh = np.frombuffer(pix.samples, dtype=np.uint8)
        self.bild = np.ascontiguousarray(
            roh.reshape(pix.height, pix.width, pix.n)[:, :, :3])
        self.faktor = dpi / befund.render_dpi
        self.dpi = dpi

    def rand(self, box: Bbox) -> float:
        """Rand in Befund-Pixeln, proportional zur Boxhöhe und beidseitig gedeckelt."""
        return float(np.clip(box.hoehe * RAND_ANTEIL, RAND_MIN_PX, RAND_MAX_PX))

    def ausschnitt(self, block: Block) -> np.ndarray:
        bx = block.bbox
        r = self.rand(bx) * self.faktor
        h, b = self.bild.shape[:2]
        x0 = int(max(0, bx.x0 * self.faktor - r))
        y0 = int(max(0, bx.y0 * self.faktor - r))
        x1 = int(min(b, bx.x1 * self.faktor + r))
        y1 = int(min(h, bx.y1 * self.faktor + r))
        return self.bild[y0:y1, x0:x1]


def als_datenurl(bild_rgb: np.ndarray) -> str:
    ok, puffer = cv2.imencode(".png", bild_rgb[:, :, ::-1])      # cv2 will BGR
    if not ok:
        raise RuntimeError("PNG-Kodierung fehlgeschlagen")
    return "data:image/png;base64," + base64.b64encode(puffer).decode("ascii")

---
## 4. Prompt und Token-Deckel

Zwei Zuordnungen, beide vom `pp_label` gesteuert.

Der **Prompt** ist eines von sechs festen Präfixen — PaddleOCR-VL nimmt keine freien
Anweisungen entgegen.

Der **Token-Deckel** ist eine Sicherung, keine Optimierung. Die Sondierung zeigte,
dass die Laufzeit fast ausschließlich an der Ausgabelänge hängt: 0,21 s für eine
zwölfstellige Fußzeile, 2,37 s für 833 Zeichen Bildunterschrift. Eine Seitenzahl, die
plötzlich 1024 Tokens erzeugt, ist keine Seitenzahl mehr, sondern eine Schleife — und
der Deckel begrenzt den Schaden auf Sekunden.

In [5]:
PROMPT_FUER: dict[str, str | None] = {
    "text": "OCR:", "paragraph_title": "OCR:", "doc_title": "OCR:",
    "abstract": "OCR:", "aside_text": "OCR:", "footnote": "OCR:",
    "vision_footnote": "OCR:", "figure_title": "OCR:", "content": "OCR:",
    "header": "OCR:", "footer": "OCR:", "number": "OCR:",
    "reference": "OCR:", "reference_content": "OCR:", "algorithm": "OCR:",
    "vertical_text": "OCR:", "formula_number": "OCR:",
    "display_formula": "Formula Recognition:",
    "inline_formula": "Formula Recognition:",
    "table": "Table Recognition:",
    "chart": "Chart Recognition:",
    "seal": "Seal Recognition:",
    "image": None, "header_image": None, "footer_image": None,
}

FORMAT_FUER: dict[str, str] = {
    "OCR:": "klartext",
    "Formula Recognition:": "latex",
    "Table Recognition:": "otsl",
    "Chart Recognition:": "datenreihe",
    "Seal Recognition:": "klartext",
}

MAX_TOKENS_FUER: dict[str, int] = {
    "number": 64, "header": 256, "footer": 256, "formula_number": 64,
    "inline_formula": 256, "display_formula": 512,
    "table": 2048, "chart": 1024,
}
MAX_TOKENS_STANDARD = 1024

print(f"{len(PROMPT_FUER)} Labels zugeordnet, "
      f"{sum(1 for p in PROMPT_FUER.values() if p is None)} ohne Aufruf")

25 Labels zugeordnet, 3 ohne Aufruf


---
## 5. Nachbereitung der Ausgabe

Drei Eingriffe, und jeder ist eine Entscheidung, die man begründen können muss.

**Inline-Formeln.** Das Modell liefert `\[…\]`, die Notation für **abgesetzte**
Mathematik — es kennt den Unterschied nicht, weil beide Fälle denselben Prompt
bekommen. Wir kennen ihn: er steht im `pp_label`. Bei `inline_formula` werden die
Delimiter zu `$…$` umgeschrieben, bei `display_formula` zu `$$…$$`.

Das ist die erste Stelle, an der sich die Entscheidung aus §5 von Notebook 1 auszahlt,
die beiden Klassen **nicht** zu einem gemeinsamen `formula` zusammenzulegen.

**OTSL bleibt OTSL.** `<fcel>`, `<ecel>`, `<lcel>`, `<ucel>`, `<xcel>`, `<nl>` sind
das Tabellenvokabular des docling-Ökosystems. Wir reichen es unverändert durch. Jede
Umwandlung hier wäre eine Interpretation zur falschen Zeit.

**Ein Wächter statt einer Reparatur.** Wenn eine Tabelle ohne `<fcel>` zurückkommt,
wird das als Warnung vermerkt und der Text trotzdem gespeichert. Nicht überschrieben,
nicht verworfen — die Entscheidung darüber gehört ans Ende der Pipeline, nicht in ihre
Mitte.

In [6]:
def nachbereiten(text: str, pp_label: str) -> tuple[str, list[str]]:
    """Rohantwort -> (bereinigter Text, Warnungen)."""
    t = text.strip()
    warnungen: list[str] = []

    if pp_label in ("inline_formula", "display_formula"):
        treffer = re.fullmatch(r"\\\[(.*?)\\\]", t, flags=re.S)
        if treffer:
            inhalt = treffer.group(1).strip()
            t = f"${inhalt}$" if pp_label == "inline_formula" else f"$$\n{inhalt}\n$$"
        elif not t.startswith("$"):
            warnungen.append(f"Formel ohne erkennbare Delimiter: {t[:40]!r}")

    if pp_label == "table" and "<fcel>" not in t and "<ecel>" not in t:
        warnungen.append("Tabelle ohne OTSL-Marken - Ausgabe unstrukturiert.")

    if not t:
        warnungen.append("Leere Antwort.")

    return t, warnungen

---
## 6. Der Lauf

`erkennen()` ist derselbe Aufruf wie in der Sondierung. `stufe2()` ist die Klammer:
zusammenführen, für jeden Block schneiden, aufrufen, nachbereiten, eintragen.

Bildblöcke bekommen keinen Aufruf. Ihr Ausschnitt landet als PNG unter
`ausschnitte/`, und der Pfad steht im Feld `ausschnitt`. Damit kann Stufe 4 sie
aufgreifen, ohne die Seite neu rendern zu müssen.

`temperature=0` bleibt: eine Pipeline, deren Ausgabe sich beim zweiten Lauf ändert,
lässt sich weder prüfen noch gegen ein Goldset messen.

In [7]:
def geladene_modelle() -> list[str]:
    antwort = requests.get(f"{LMS_URL}/models", timeout=10)
    antwort.raise_for_status()
    return [m["id"] for m in antwort.json()["data"]]


if MODELL_ID is None:
    MODELL_ID = geladene_modelle()[0]
print("Modell:", MODELL_ID)


def erkennen(bild_rgb: np.ndarray, prompt: str, max_tokens: int) -> tuple[str, float]:
    nutzlast = {
        "model": MODELL_ID,
        "temperature": 0,
        "max_tokens": max_tokens,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": als_datenurl(bild_rgb)}},
                {"type": "text", "text": prompt},
            ],
        }],
    }
    t0 = time.perf_counter()
    antwort = requests.post(f"{LMS_URL}/chat/completions", json=nutzlast, timeout=300)
    dauer = time.perf_counter() - t0
    antwort.raise_for_status()
    return antwort.json()["choices"][0]["message"]["content"], dauer


def stufe2(name: str, zeige_fortschritt: bool = True) -> SeitenBefund:
    """Befund der Stufe 1 -> Befund mit gefülltem Feld `text`."""
    roh = json.loads((BEFUND_DIR / f"{name}_stufe1.json").read_text(encoding="utf-8"))
    befund = zusammenfuehren(SeitenBefund.model_validate(roh))
    seite = Seitenbild(SEITEN[name], befund)

    gesamt = 0.0
    for blk in befund.bloecke:
        aus = seite.ausschnitt(blk)
        prompt = PROMPT_FUER.get(blk.pp_label)

        if prompt is None:                       # Bildblock: nur ablegen
            ziel = AUSSCHNITT_DIR / f"{name}_{blk.id:02d}_{blk.pp_label}.png"
            cv2.imwrite(str(ziel), aus[:, :, ::-1])
            blk.ausschnitt = str(ziel)
            if zeige_fortschritt:
                print(f"  #{blk.id:2d} {blk.pp_label:18s} -> {ziel.name}")
            continue

        text, dauer = erkennen(
            aus, prompt, MAX_TOKENS_FUER.get(blk.pp_label, MAX_TOKENS_STANDARD))
        gesamt += dauer

        text, warnungen = nachbereiten(text, blk.pp_label)
        blk.text = text
        blk.text_format = FORMAT_FUER[prompt]
        blk.text_quelle = "paddleocr_vl"
        befund.warnungen += [f"#{blk.id}: {w}" for w in warnungen]

        if zeige_fortschritt:
            vorschau = text[:58].replace("\n", " ")
            print(f"  #{blk.id:2d} {blk.pp_label:18s} {dauer:5.2f}s "
                  f"{len(text):5d}z  {vorschau}")

    befund.warnungen.append(f"Stufe 2: {gesamt:.1f} s Modellzeit.")
    return befund

Modell: mlx-community/paddleocr-vl-1.5


In [8]:
NAME = "zimbardo"          # ANPASSEN

t0 = time.perf_counter()
BEFUND2 = stufe2(NAME)
print(f"\n{len(BEFUND2.bloecke)} Blöcke, {time.perf_counter() - t0:.1f} s gesamt")

print("\nWarnungen:")
for w in BEFUND2.warnungen:
    print(f"  {w}")

  # 0 header              0.70s    36z  Sensorische Prozesse und Wahrnehmung
  # 1 table               2.80s   305z  <fcel>A. Eben merklich längere Striche<fcel>B. Standard-st
  # 2 inline_formula      0.24s    22z  $L_{a}-L_{b}=\Delta L$
  # 3 chart               2.04s   122z  Standardstrichlänge (L_b) in Millimetern | Unterschiedssch
  # 4 figure_title        2.36s   833z  Abbildung 4.8: Eben merkliche Unterschiede und Weber‘sches
  # 5 figure_title        0.27s    53z  Weber‘sche Konstanten für ausgewählte Reizdimensionen
  # 6 table               1.21s   279z  <fcel>Reizdimension<fcel>Weber’sche Konstante (k)<nl><fcel
  # 7 text                0.91s   298z  Töne besser unterscheiden kann als die Intensität zweier L
  # 8 paragraph_title     0.26s    48z  4.2.2 Von physikalischen zu mentalen Ereignissen
  # 9 text                0.86s   291z  Unser Überblick über die Psychophysik hat Sie vielleicht a
  #10 footer              0.37s    77z  118 Persönliches Exemplar von Herr Hans Wur

---
## 7. Kontrolle

Die billigste Prüfung ist auch hier ein Blick auf das Ergebnis — diesmal nicht als
Bild, sondern als Markdown. Wenn der Hauptstrom sich flüssig lesen lässt, stimmen
Lesereihenfolge, Zuschnitt und Erkennung zugleich. Wenn nicht, sieht man meist
sofort, welche der drei nicht stimmt.

Die Marginalien und die Boilerplate kommen getrennt darunter. Sie gehören nicht in
den Lesefluss — genau das war die Aussage der Ströme aus Notebook 1, und hier wird
sie zum ersten Mal sichtbar.

In [9]:
def als_markdown(befund: SeitenBefund, strom: Strom = Strom.HAUPT) -> str:
    """Grobe Vorschau. Die richtige Umwandlung ist Stufe 4 - hier geht es ums Prüfen."""
    zeilen: list[str] = []
    for blk in befund.lesefolge(strom):
        if blk.ausschnitt:
            zeilen.append(f"![{blk.pp_label}]({blk.ausschnitt})")
        elif blk.text is None:
            zeilen.append(f"<!-- #{blk.id} {blk.pp_label}: kein Text -->")
        elif blk.pp_label == "doc_title":
            zeilen.append(f"# {blk.text}")
        elif blk.pp_label == "paragraph_title":
            zeilen.append(f"## {blk.text}")
        elif blk.pp_label == "figure_title":
            zeilen.append(f"*{blk.text}*")
        elif blk.text_format in ("otsl", "datenreihe"):
            zeilen.append(f"```{blk.text_format}\n{blk.text}\n```")
        else:
            zeilen.append(blk.text)
        zeilen.append("")
    return "\n".join(zeilen)


from IPython.display import Markdown, display

for strom in (Strom.HAUPT, Strom.MARGINALIE, Strom.APPARAT, Strom.BOILERPLATE):
    inhalt = als_markdown(BEFUND2, strom)
    if not inhalt.strip():
        continue
    print("=" * 70)
    print(f"### Strom: {strom.value}")
    display(Markdown(inhalt))

### Strom: haupt


```otsl
<fcel>A. Eben merklich längere Striche<fcel>B. Standard-strichlänge<fcel>L_{a} - L_{b} = \(\Delta L\)<nl><fcel>11 mm<fcel>10 mm<fcel>11,0 - 10,0 = 1,0<nl><fcel>16,5 mm<fcel>15 mm<fcel>16,5 - 15,0 = 1,5<nl><fcel>22 mm<fcel>20 mm<fcel>22,0 - 20,0 = 2,0<nl><fcel>27,5 mm<fcel>25 mm<fcel>27,5 - 25,0 = 2,5<nl>
```

$L_{a}-L_{b}=\Delta L$

```datenreihe
Standardstrichlänge (L_b) in Millimetern | Unterschiedsschwelle (ΔL/L_b = 0,1)
6 | 0.5
10 | 1.0
15 | 1.5
20 | 2.0
25 | 2.5
```

*Abbildung 4.8: Eben merkliche Unterschiede und Weber‘sches Gesetz. Angenommen, Sie führen ein Experiment durch, dessen Teilnehmer angeben sollen, ob zwei Striche gleich oder verschieden lang sind. Je länger der Referenzstrich, desto größer ist die hinzu-zufügende Länge (ΔL) von Vergleichsstrichen, um einen eben merklichen Unterschied zu erzielen. Die Unterschiedsschwelle ist die jeweils zu addierende Länge, damit in je der Hälfte der Fälle ein Unterschied erkannt wird. Wenn dieser jeweilige Zuwachs gegen die steigende Länge der Referenzstriche abgetragen wird, bleibt der Quotient gleich. Der benötigte Zuwachs ist immer ein Zehntel der Standardlänge. Die Beziehung ist linear, was sich in der Grafik als Gerade ausdrückt. Wir können vorhersagen, dass ∆L bei einem Referenzstrich von 5 Millimetern 0,5 Millimeter betragen wird.*

*Weber‘sche Konstanten für ausgewählte Reizdimensionen*

```otsl
<fcel>Reizdimension<fcel>Weber’sche Konstante (k)<nl><fcel>Schallfrequenz<fcel>0,003<nl><fcel>Lichtintensität<fcel>0,01<nl><fcel>Geruchs-konzentration<fcel>0,07<nl><fcel>Druckintensität<fcel>0,14<nl><fcel>Schallintensität<fcel>0,15<nl><fcel>Geschmacks-konzentration<fcel>0,20<nl>
```

Töne besser unterscheiden kann als die Intensität zweier Lichtpunkte. Diese wiederum sind durch einen kleineren EMU besser zu entdecken als Geruchs- oder Geschmacksunterschiede. Ihre Getränkefabrik bräuchte eine relativ große Menge zusätzlichen Zuckers, um eine merklich süßere Cola zu produzieren!

## 4.2.2 Von physikalischen zu mentalen Ereignissen

Unser Überblick über die Psychophysik hat Sie vielleicht auf das grundlegende Mysterium von Empfindungen aufmerksam gemacht: Wie werden aus physikalischen Energien spezifische psychische Erfahrungen? Wie entsteht beispielsweise aus Licht unterschiedlicher physikalischer Wellenlängen die Er-


### Strom: boilerplate


Sensorische Prozesse und Wahrnehmung

118
Persönliches Exemplar von Herr Hans Wurst vom 27.04.2011, Lesen & Drucken


In [10]:
# --- Ablegen: die Schnittstelle zu Stufe 4
ziel = BEFUND_DIR / f"{NAME}_stufe2.json"
ziel.write_text(BEFUND2.model_dump_json(indent=2), encoding="utf-8")
print(f"{ziel}  {ziel.stat().st_size / 1024:.1f} kB")

verteilung: dict[str, int] = {}
for blk in BEFUND2.bloecke:
    schluessel = blk.text_format or ("bild" if blk.ausschnitt else "leer")
    verteilung[schluessel] = verteilung.get(schluessel, 0) + 1
print("Formate:", verteilung)

befunde/zimbardo_stufe2.json  12.3 kB
Formate: {'klartext': 7, 'otsl': 2, 'latex': 1, 'datenreihe': 1}


---
## 8. Offene Punkte

| # | Punkt | Wie zu klären |
|---|---|---|
| 1 | Die Schwellen `MAX_LUECKE_PX` und `MIN_UEBERLAPP` | am Goldset justieren; derzeit geraten, nicht gemessen |
| 2 | Zusammenführen über Zeilengrenzen | die Regel wirkt nur waagerecht. Ein zweispaltig zerschnittener Absatz bleibt zerschnitten |
| 3 | Überlappende Blöcke | Tabelle und Diagramm der Zimbardo-Seite liegen ineinander; beide werden erkannt, der Inhalt doppelt sich |
| 4 | `RAND_ANTEIL` | 12 % ist gesetzt, nicht optimiert. Zu prüfen an Blöcken mit Unterlängen und Formelklammern |
| 5 | Der Scan | für Tietze-Schenk ist das VLM die **einzige** Quelle. Ohne zweite Quelle gibt es keine Verankerung — der Kern von Phase 05 |
| 6 | Wiederholte Läufe | `temperature=0` macht die Ausgabe stabil, aber nicht garantiert identisch über Modellversionen hinweg. Für ein Goldset gehört die Modellversion in die Metadaten |

Zu Punkt 3: Das Zusammenführen hilft hier nicht, im Gegenteil — die Regel lässt
Tabellen und Diagramme bewusst unberührt. Zwei ineinanderliegende Blöcke sind kein
Zuschnittsproblem, sondern zwei konkurrierende Kandidaten. Genau dafür hält das
Zwischenschema `score` und `quelle`, und genau das löst Stufe 4 auf.

Zu Punkt 6: `quelle_datei`, `render_dpi` und die Blockgeometrie stehen im Befund, die
verwendete Modellversion nicht. Für Reproduzierbarkeit fehlt sie — ein Feld
`modell` im `SeitenBefund` wäre die naheliegende Ergänzung, sobald mehr als eine
Version im Spiel ist.

---

## 9. Übungsaufgaben

**A — Die Schwellen begründen (mittel).**
`MAX_LUECKE_PX = 40` entspricht bei 200 dpi etwa 14 pt. Rechnen Sie nach, wie breit
ein Leerzeichen in einer 10-pt-Schrift ist, und begründen Sie, warum der Wert
darüber, aber deutlich unter einem Spaltenabstand liegen muss.

**B — Zusammenführen prüfen (mittel).**
Lassen Sie das Notebook einmal mit `MAX_LUECKE_PX = 0` laufen und einmal mit `200`.
Vergleichen Sie die Fußzeile im Strom `boilerplate`. Was passiert im zweiten Fall mit
der Marginalspalte der Wahrnehmungspsychologie-Seite, und warum verhindert die
Strom-Bedingung den Schaden nicht vollständig?

**C — Formate ernst nehmen (fortgeschritten).**
Schreiben Sie eine Funktion, die `text_format == "otsl"` in eine Markdown-Tabelle
umwandelt. Welche OTSL-Marken lassen sich dabei **nicht** abbilden, und was folgt
daraus für die Wahl von Markdown als Zielformat?

**D — Verankerung (Projektaufgabe).**
Für die born-digital-Seiten liefert PyMuPDF eine zweite Quelle. Erweitern Sie den
Befund um ein Feld `text_pymupdf` und eine Ähnlichkeit je Block. Ab welchem Wert
würden Sie einen Block zur Handprüfung markieren — und wie begründen Sie die
Schwelle gegenüber jemandem, der 5000 Seiten verarbeiten will?